# What's in a Name — Endonyms and Exonyms

What do languages call themselves, and what does English call them? This notebook
explores cross-lingual naming via `db.language_names` and each language's
`.endonym` property.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [1]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [2]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Canonical name records: {len(db.language_names):,}")

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)
Canonical name records: 167,917


## 2 · Endonym coverage

An endonym is a name expressed in the language itself (`LanguageName.is_endonym`).

In [3]:
with_endonym = sum(1 for lang in db.languages if lang.endonym)
total = len(db.languages)
print(f"Languages with a known endonym: {with_endonym:,} / {total:,} ({100 * with_endonym / total:.1f}%)")

for part3 in ("deu", "hin", "kin", "jpn"):
    lang = db.languages.get(part3)
    endo = lang.endonym.name if lang.endonym else "—"
    print(f"  {lang.label} ({part3}): endonym = {endo}")

Languages with a known endonym: 1,137 / 7,929 (14.3%)
  German (deu): endonym = Deutsch
  Hindi (hin): endonym = हिन्दी
  Kinyarwanda (kin): endonym = Ikinyarwanda
  Japanese (jpn): endonym = 日本語


## 3 · English exonyms via `in_language`

In [4]:
english_names = db.language_names.in_language("en")
print(f"English name records: {len(english_names):,}")

en_df = pd.DataFrame([
    {
        "part3": n.language.part3,
        "language": n.language.label,
        "english_name": n.name,
        "endonym": n.language.endonym.name if n.language.endonym else None,
    }
    for n in english_names
]).drop_duplicates(subset=["part3"])

en_df.head(10)

English name records: 7,494


,part3,language,english_name,endonym
0,aaa,Ghotuo,Ghotuo,NaN
1,aab,Alumu-Tesu,Alumu-Tesu,NaN
2,aac,Ari,Ari,NaN
3,aad,Amal,Amal,NaN
4,aae,Arbëreshë Albanian,Arbëreshë Albanian,NaN
5,aaf,Aranadan,Aranadan,NaN
6,aag,Ambrak,Ambrak,NaN
7,aah,Abu' Arapesh,Abu' Arapesh,NaN
8,aai,Arifama-Miniafia,Arifama-Miniafia,NaN
9,aak,Ankave,Ankave,NaN


## 4 · Coverage bar chart

In [5]:
coverage_rows = [
    {"category": "Has endonym", "count": with_endonym},
    {"category": "No endonym", "count": total - with_endonym},
]
cov_df = pd.DataFrame(coverage_rows)

fig = px.bar(
    cov_df,
    x="category",
    y="count",
    text="count",
    labels={"count": "Languages", "category": ""},
    title="Endonym Coverage Across All Languages",
    color="category",
    color_discrete_sequence=["#4c78a8", "#e45756"],
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · What is German called in French?

Use `for_language` to list all canonical names for one language.

In [6]:
deu = db.languages.get("deu")
print(f"All canonical names for {deu.label}:")
for n in deu.names:
    endo_tag = " (endonym)" if n.is_endonym else ""
    print(f"  [{n.in_language_bcp47}] {n.name}{endo_tag}")

french_names = [n.name for n in deu.names if n.in_language_bcp47 == "fr"]
print(f"\nFrench exonym(s): {french_names}")

All canonical names for German:
  [ab] агерман
  [ace] Bahsa Jeureuman
  [ady] Германыбзэ
  [aeb] الألماني
  [af] Duits
  [agq] Dzamɛ̀
  [ak] Gyaaman
  [am] ጀርመን
  [an] alemán
  [ang] Þēodsc sprǣc
  [anp] जर्मन भाषा
  [ar] الألمانية
  [arc] ܠܫܢܐ ܓܪܡܢܝܐ
  [arq] الألمانية
  [ary] لألمانية
  [arz] لغه المانى
  [as] জাৰ্মান
  [asa] Kijerumani
  [ast] alemán
  [av] Герман мацӀ
  [ay] Aliman aru
  [az] alman
  [azb] آلمان دیلی
  [ba] Немец теле
  [bal] جرمن
  [ban] Basa Jerman
  [bar] deitsche Sproch
  [bas] Hɔp u jamân
  [bcl] Aleman
  [be] нямецкая
  [bem] Ichi Jemani
  [bez] Hijerumani
  [bg] немски
  [bgn] جرمنی
  [bho] जर्मन भाषा
  [bm] alimaɲikan
  [bn] জার্মান
  [bo] འཇར་མན་གྱི།
  [bpy] জার্মান ঠার
  [br] alamaneg
  [brx] जार्मान
  [bs] njemački
  [bxr] Герман хэлэн
  [byn] ጀርመን
  [ca] alemany
  [ccp] 𑄎𑄢𑄴𑄟𑄚𑄴
  [cdo] Dáik-ngṳ̄
  [ce] немцойн
  [ceb] German
  [cgg] Orugirimaani
  [chr] ᏙᎢᏥ
  [chy] Ma'ėhoéseotse
  [ckb] ئەڵمانی
  [cmn] 德语
  [co] tedescu
  [crh] Alman tili
  [cs] němčina


## 6 · European language name matrix

In [7]:
EUROPEAN = ["deu", "fra", "ita", "spa", "eng", "nld", "pol", "rus"]
MATRIX_LANGS = ["en", "de", "fr"]

matrix_rows = []
for part3 in EUROPEAN:
    lang = db.languages.get(part3)
    row = {"language": lang.endonym.name if lang.endonym else lang.label}
    for bcp47 in MATRIX_LANGS:
        names = [n.name for n in lang.names if n.in_language_bcp47 == bcp47]
        row[bcp47] = names[0] if names else "—"
    matrix_rows.append(row)

matrix_df = pd.DataFrame(matrix_rows)
matrix_df

,language,en,de,fr
0,Deutsch,German,Deutsch,allemand
1,français,French,Französisch,français
2,italiano,Italian,Italienisch,italien
3,español,Spanish,Spanisch,espagnol
4,English,English,Englisch,anglais
5,Nederlands,Dutch,Niederländisch,néerlandais
6,polski,Polish,Polnisch,polonais
7,русский,Russian,Russisch,russe


## 7 · Bulk endonym list

In [8]:
endonyms = db.language_names.endonyms()
print(f"Total endonym records: {len(endonyms):,}")

endo_df = pd.DataFrame([
    {"part3": n.language.part3, "language": n.language.label, "endonym": n.name}
    for n in endonyms
]).drop_duplicates(subset=["part3"]).sort_values("language")

endo_df.head(15)

Total endonym records: 1,138


,part3,language,endonym
5,abq,Abaza,абаза
3,abk,Abkhazian,Аҧсшәа
6,abr,Abron,Brong
1,aba,Abé,Abé
9,ace,Achinese,Acèh
14,act,Achterhoeks,Achterhoeks
11,ach,Acoli,Acholi
17,ada,Adangme,Dangbe
458,kad,Adara,Eda
18,adh,Adhola,Dhopadhola


## 8 · Summary

In [9]:
print(f"Languages with English name: {en_df['part3'].nunique():,}")
print(f"Languages with endonym: {with_endonym:,}")
both = en_df.merge(endo_df, on="part3", how="inner")
print(f"Languages with both English name and endonym: {len(both):,}")

Languages with English name: 7,494
Languages with endonym: 1,137
Languages with both English name and endonym: 1,137
